# encoder
# decoder
# train
# generate

In [8]:
import torch
import torch.nn as nn

In [9]:
chars = "0123456789-/"
special_tokens = ["<SOS>", "<EOS>"]

itos = special_tokens + list(chars)
stoi = {ch: i for i, ch in enumerate(itos)}

SOS_token = stoi["<SOS>"]
EOS_token = stoi["<EOS>"]

vocab_size = len(itos)

print(itos)
print(stoi)

['<SOS>', '<EOS>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '/']
{'<SOS>': 0, '<EOS>': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, '-': 12, '/': 13}


In [10]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.1)

        # 你来写 embedding
        # 你来写 rnn

    def forward(self, input):
        # 你来写一次前向传播
        input = self.embedding(input)
        input = self.dropout(input)
        input = input.unsqueeze(1)
        output, hidden = self.rnn(input)
        return output, hidden


In [11]:
import random
def make_sample():
    ep = 100
    inputs = []
    targets = []
    for i in range(ep):
        year = random.randint(1950, 2050)
        day = random.randint(1, 28)
        month = random.randint(1, 12)
        input = f"{year}-{month}-{day}"
        inputs.append(input)
        target = f"{day}/{month}/{year}"
        targets.append(target)
    return inputs, targets

inputs, targets = make_sample()
print(inputs[0:10])
print(targets[0:10])


['1989-7-5', '2034-3-11', '2006-1-28', '2005-9-23', '2002-11-11', '2035-12-28', '1983-9-16', '2023-11-2', '1967-5-17', '2035-6-12']
['5/7/1989', '11/3/2034', '28/1/2006', '23/9/2005', '11/11/2002', '28/12/2035', '16/9/1983', '2/11/2023', '17/5/1967', '12/6/2035']


In [13]:
encoder = EncoderRNN(vocab_size, 5)
input_tensor = torch.tensor([stoi[ch] for ch in inputs[0]], dtype=torch.long)
print("input_tensor",input_tensor)
output,hidden = encoder.forward(input_tensor)
print(output,hidden)

input_tensor tensor([ 3, 11, 10, 11, 12,  9, 12,  7])
tensor([[[-0.6802, -0.0858, -0.3551, -0.3331,  0.6981]],

        [[-0.7198, -0.1559, -0.6150, -0.3374,  0.7410]],

        [[ 0.5056, -0.9036, -0.1862, -0.3661, -0.4759]],

        [[-0.7246, -0.2168, -0.6286, -0.2759,  0.7377]],

        [[ 0.4435, -0.8897,  0.0463,  0.1259, -0.2308]],

        [[ 0.4241, -0.8320, -0.0668, -0.0748, -0.1542]],

        [[ 0.4435, -0.8897,  0.0463,  0.1259, -0.2308]],

        [[-0.0938, -0.3778,  0.3704,  0.3858,  0.5565]]],
       grad_fn=<TransposeBackward1>) tensor([[[-0.6802, -0.0858, -0.3551, -0.3331,  0.6981],
         [-0.7198, -0.1559, -0.6150, -0.3374,  0.7410],
         [ 0.5056, -0.9036, -0.1862, -0.3661, -0.4759],
         [-0.7246, -0.2168, -0.6286, -0.2759,  0.7377],
         [ 0.4435, -0.8897,  0.0463,  0.1259, -0.2308],
         [ 0.4241, -0.8320, -0.0668, -0.0748, -0.1542],
         [ 0.4435, -0.8897,  0.0463,  0.1259, -0.2308],
         [-0.0938, -0.3778,  0.3704,  0.3858,  0.5565

In [6]:
print(output.shape)
print(hidden.shape)

torch.Size([9, 1, 5])
torch.Size([1, 1, 5])


In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_output, target_tensor):
        # encoder_output是encoder输出的hiddeng向量
        # target_tensor是decoder输出的向量
        last_target_tensor #记住上一次生成的词元id
        hidden = encoder_output
        input_token = SOS_token #设置起始词元

        outputs = []
        for i in range(12):

            self.forward_step(hidden, input_token)
            if target_tensor is not None:
                input.append(target_tensor[i:i+1])
           
            embedded = self.embedding(input)
            rnn_output, hidden = nn.rnn(embedded, hidden)
            decode_ouput = self.out(rnn_output)
            predict = decode_ouput.argmax(dim = -1, keepdim=True)
            outputs.append(predict)
            last_target_tensor = self.embedding(predict)
            if target_tensor is None:
                input = predict
        return torch.cat(outputs, dim=1)  # (batch, max_len)
        
    def forward_step(self, hidden, input_token):
        # input_token.shape是(batch_size,seq_len),元素内容是词元id
        # embedded.shape应当是(batch_size,seq_len,hidden)，元素内容是次元id对应的embedding值
        embedded = self.embedding(input_token)
        # rnn_output 是所有时间步骤的最后一层,shape是 (batch_size,seq_len,hidden)
        # hidden 是最后一个时间步骤的所有层,shape是 (batch_size, num_layers * num_directions,hidden),本例中 num_layers * num_directions 是1
        rnn_output, hidden = self.rnn(embedded, hidden) 
        output = self.out(hidden[-1]) # out 的输入是最后一维是hidden的任何形状，输出是将输入的最后一维改成和Linear初始化设置的输出一样的形状
        predictIdx = output.argmax(dim = -1, keepdim=True)
        return predictIdx, hidden

    

In [ ]:
hidden_size = 5
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)


In [11]:
print(decoder.out.shape)

AttributeError: 'Linear' object has no attribute 'shape'